# Fine-Tuning PhoBERT Multi-Label Moderation (v1.1 - Undersample CLEAN)

Notebook này nạp tập dữ liệu kiểm duyệt **v1.1 (Undersampled CLEAN = 8,000 mẫu)** đã phân chia theo tỷ lệ học sâu chuẩn **70% Train (11,389 mẫu)**, **15% Val (2,440 mẫu)**, và **15% Test (2,441 mẫu)** từ `data/v1.1/` (hoặc Root working directory trên Google Colab).

### 🏷️ Taxonomy 4 Nhãn AI Ngữ Cảnh:
- `0: CLEAN` - Nội dung an toàn, tích cực.
- `1: PROFANITY_VENTING` - Từ chửi thề nhẹ / Bộc phát xả stress.
- `2: HATE_SPEECH` - Ngôn từ thù ghét / Công kích cá nhân.
- `3: EMOTIONAL_CRISIS` - Khủng hoảng cảm xúc / Bế tắc / Trầm cảm / Ý định tự hại.
*(Lưu ý: Nhãn `4: ILLEGAL_PORN` được đảm bảo bởi Lớp 1 Hard-Block Engine bằng Regex/Rules)*

In [ ]:
# Step 1: Kiểm tra môi trường & Cài đặt các thư viện cần thiết
!pip install -q transformers datasets torch accelerate scikit-learn matplotlib seaborn

In [ ]:
# Step 2: Nạp các tập train.json, val.json, test.json của phiên bản v1.1
import os
import json
from pathlib import Path

def find_v1_1_data_file(filename):
    candidates = [
        Path(filename),                             # Direct root (Colab)
        Path("v1.1") / filename,                    # Colab v1.1 subfolder
        Path("../data/v1.1") / filename,            # Relative pipeline v1.1
        Path("../../data/v1.1") / filename,         # Relative version folder v1.1
        Path("moderation/data/v1.1") / filename     # Project root relative
    ]
    for p in candidates:
        if p.exists():
            return p
    return Path(filename)

train_path = find_v1_1_data_file("train.json")
val_path = find_v1_1_data_file("val.json")
test_path = find_v1_1_data_file("test.json")

with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(val_path, "r", encoding="utf-8") as f:
    val_data = json.load(f)
with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Loaded v1.1 Train samples from {train_path}: {len(train_data):,}")
print(f"Loaded v1.1 Val samples   from {val_path}  : {len(val_data):,}")
print(f"Loaded v1.1 Test samples  from {test_path} : {len(test_data):,}")

In [ ]:
# Step 3: Tokenization & Chuyển đổi sang định dạng HuggingFace Dataset
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix

MODEL_NAME = "vinai/phobert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_ds = Dataset.from_dict({"text": [x["text"] for x in train_data], "label": [x["label"] for x in train_data]}).map(tokenize_fn, batched=True)
val_ds = Dataset.from_dict({"text": [x["text"] for x in val_data], "label": [x["label"] for x in val_data]}).map(tokenize_fn, batched=True)
test_ds = Dataset.from_dict({"text": [x["text"] for x in test_data], "label": [x["label"] for x in test_data]}).map(tokenize_fn, batched=True)

# Khởi tạo mô hình PhoBERT-base-v2 cho 4 nhãn kiểm duyệt ngữ cảnh
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4)

In [ ]:
# Step 4: Cấu hình tham số huấn luyện (TrainingArguments) & Early Stopping theo dõi eval_loss chống Overfitting
eval_strat_key = "eval_strategy" if hasattr(TrainingArguments, "eval_strategy") else "evaluation_strategy"

args_dict = {
    "output_dir": "./results_phobert_moderation_v1.1",
    "num_train_epochs": 5,
    "per_device_train_batch_size": 32,
    "per_device_eval_batch_size": 32,
    "learning_rate": 2e-5,
    "warmup_steps": 300,
    "weight_decay": 0.01,
    eval_strat_key: "epoch",
    "save_strategy": "epoch",
    "save_total_limit": 5,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "fp16": torch.cuda.is_available(),
    "logging_steps": 50
}

training_args = TrainingArguments(**args_dict)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": macro_f1, "weighted_f1": weighted_f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
# Step 5: Bắt đầu Huấn luyện Mô hình PhoBERT Moderation v1.1
print("Starting PhoBERT Multi-Label Moderation Fine-Tuning (v1.1)...")
trainer.train()

In [ ]:
# Step 6: ĐÁNH GIÁ VÀ TỰ ĐỘNG TÌM CHECKPOINT CÓ MACRO F1 TỐT NHẤT TRÊN TẬP TEST
import glob
from transformers import AutoModelForSequenceClassification

target_names = ["0: CLEAN", "1: PROFANITY_VENTING", "2: HATE_SPEECH", "3: EMOTIONAL_CRISIS"]
checkpoint_dirs = sorted(glob.glob("./results_phobert_moderation_v1.1/checkpoint-*"))

best_checkpoint_path = None
best_test_macro_f1 = -1.0
best_test_report = ""
best_test_preds = None
best_test_labels = None

print(f"Tìm thấy {len(checkpoint_dirs)} checkpoints để đánh giá đối chứng trên tập Test:")
for ckpt in checkpoint_dirs:
    eval_model = AutoModelForSequenceClassification.from_pretrained(ckpt)
    eval_trainer = Trainer(
        model=eval_model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics
    )
    
    test_preds = eval_trainer.predict(test_ds)
    p_labels = test_preds.label_ids
    p_preds = np.argmax(test_preds.predictions, axis=1)
    
    m_f1 = f1_score(p_labels, p_preds, average="macro")
    w_f1 = f1_score(p_labels, p_preds, average="weighted")
    acc = accuracy_score(p_labels, p_preds)
    report_str = classification_report(p_labels, p_preds, target_names=target_names, digits=4)
    
    print(f"📌 CHECKPOINT {ckpt}: Macro F1 = {m_f1*100:.2f}% | Weighted F1 = {w_f1*100:.2f}% | Accuracy = {acc*100:.2f}%")
    
    if m_f1 > best_test_macro_f1:
        best_test_macro_f1 = m_f1
        best_checkpoint_path = ckpt
        best_test_report = report_str
        best_test_preds = p_preds
        best_test_labels = p_labels

print("\n" + "="*75)
print(f"🏆 CHECKPOINT XUẤT SẮC NHẤT TRÊN TẬP TEST: {best_checkpoint_path}")
print(f"🏆 ĐẠT TEST MACRO F1 = {best_test_macro_f1*100:.2f}%")
print("="*75 + "\n")
print(best_test_report)

In [ ]:
# Step 7: Vẽ Confusion Matrix v1.1 cho checkpoint xuất sắc nhất
import matplotlib.pyplot as plt
import seaborn as sns

if best_test_labels is not None and best_test_preds is not None:
    cm = confusion_matrix(best_test_labels, best_test_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=target_names, yticklabels=target_names)
    plt.title(f"Confusion Matrix - Best Checkpoint ({best_checkpoint_path.split('/')[-1]})")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.show()

In [ ]:
# Step 8: Tự động tải & lưu Checkpoint XUẤT SẮC NHẤT vào thư mục ./saved_phobert_moderation_v1.1
print(f"Đang lưu mô hình xuất sắc nhất từ {best_checkpoint_path} vào ./saved_phobert_moderation_v1.1...")
best_model = AutoModelForSequenceClassification.from_pretrained(best_checkpoint_path)

save_directory = "./saved_phobert_moderation_v1.1"
best_model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"SUCCESS! Best Model ({best_checkpoint_path}) successfully saved to {save_directory}")

In [ ]:
# Step 9: Đóng gói ZIP và Tải trực tiếp v1.1 về máy tính
import shutil
from google.colab import files

zip_name = "saved_phobert_moderation_v1.1"
shutil.make_archive(zip_name, 'zip', save_directory)
print(f"Created {zip_name}.zip successfully!")

# Tải file zip v1.1 về máy tính
files.download(f"{zip_name}.zip")